# Learning-numbers report

Self-contained companion to `Amery_Future_Learning_Plots.ipynb` — **only the new number-reporting cells**. Reproduces and extends every headline learning number for any of the four error-model configurations (A `indep_code`, B `wiener_code`, C `indep_nocode`, D `wiener_nocode`), straight from the cached width files in `outputs/{posteriors,projections}_<tag>/`. Run the setup cell, then either report cell. Run from the `notebooks/` directory (paths are relative to it).

**Prerequisite:** run `Amery_Future_Learning_Plots.ipynb` first (once per configuration you want to report) to generate the cached `total_widths.jld2` and `slr_learning_widths.jld2` under `outputs/{posteriors,projections}_<tag>/`. This notebook only *reads* those caches — it does not recompute them (`sle_best` also reads the per-year `pred_VAF.jld2` projection files).


In [ ]:
# ---- setup (imports, configuration switch, constants) ----
using JLD2, Statistics, Printf

SCEN = Dict("A" => "_indep_code", "B" => "_wiener_code",
            "C"  => "_indep_nocode", "D" => "_wiener_nocode")
CONFIG = "D"                                    # ← pick one: "A", "B", "C", "D"
SUFFIX   = SCEN[CONFIG]
POST, PROJ = "../outputs/posteriors$(SUFFIX)", "../outputs/projections$(SUFFIX)"

PNAMES = ["σ_max", "q", "C_μ", "C_φ", "log(γ₀)", "m̄"]   # column order; γ₀ (col 5) in log-space
SLOTS  = [2015; collect(2030:15:2300)]                    # 20 analysis-year slots (2015,2030,…,2300)
PRED   = [2100, 2200, 2300]
REAL   = 1:100
hrow(py) = py - 2015                                      # horizon row on the 2016–2300 (285-step) axis
println("configuration $CONFIG  →  $SUFFIX")

## Current-configuration report
Set `CONFIG` in the setup cell, then run this. Parameter + SLE learning (widths, folds, %reduction, half-learning year) recomputed from that configuration's caches, plus query helpers `param_width(p,y)`, `param_fold(p)`, `sle_width(h,y)`, `sle_best(h,y)` to pull any single value.


In [ ]:
# ---- CURRENT-configuration report (loads this configuration's caches directly) ----
tw = JLD2.load("$POST/total_widths.jld2", "total_widths")        # 100 × 20 × 6
MW = [mean(filter(!isnan, tw[:, i, j])) for i in 1:size(tw, 2), j in 1:6]
pbnd(i, j) = quantile(filter(!isnan, tw[:, i, j]), [0.17, 0.83])
slr = JLD2.load("$PROJ/slr_learning_widths.jld2")
present_width, slr_widths = slr["present_width"], slr["slr_widths"]

println("CONFIG = $CONFIG   ($SUFFIX)")
println("\nPARAMETER LEARNING — 90% CI width (γ₀ in log-space);  fold = width(2015)/width(2300)")
@printf("%-9s %12s %16s %8s %7s %8s\n", "param", "2015", "2300 [lo,hi]", "fold", "%red", "half-yr")
for j in 1:6
    b0 = MW[1, j];  bf = MW[end, j];  lo, hi = pbnd(length(SLOTS), j)
    half = b0 - 0.5*(b0 - bf)
    hs   = findfirst(s -> MW[s, j] <= half, 1:length(SLOTS));  hy = hs === nothing ? 0 : SLOTS[hs]
    @printf("%-9s %12.4g %7.3g[%.3g,%.3g] %8.1f %6.1f%% %8d\n", PNAMES[j], b0, bf, lo, hi, b0/bf, 100*(1 - bf/b0), hy)
end

println("\nSEA-LEVEL LEARNING — 90% CI width (mm SLE);  fold = width(2015)/width(final)")
@printf("%-8s %10s %18s %8s %7s\n", "horizon", "2015", "final [lo,hi]", "fold", "%red")
for py in PRED
    b0 = present_width[py];  W = slr_widths[py];  fc = filter(!isnan, W[:, end])
    bf = mean(fc);  lo, hi = quantile(fc, [0.17, 0.83]);  finyr = 2015 + 15*(size(W, 2) - 1)
    @printf("%-8d %10.3f %7.3f[%.2f,%.2f]@%d %8.2f %6.1f%%\n", py, b0, bf, lo, hi, finyr, b0/bf, 100*(1 - bf/b0))
end

# ---- query any single value (current configuration) ----
param_width(p, y) = MW[findfirst(==(y), SLOTS), p]          # param p (1..6), analysis year y
param_fold(p)     = MW[1, p] / MW[end, p]                   # 2015→2300 fold for param p
function sle_width(h, y)                                    # SLE width (mm) at horizon h, calibrating through y
    W = slr_widths[h];  yrs = collect(range(2015, step=15, length=size(W, 2)))
    mean(filter(!isnan, W[:, findfirst(==(y), yrs)]))
end
function sle_best(h, y)                                     # SLE best estimate (mm) — loads projection files
    ms = Float64[]
    for r in REAL
        f = "$PROJ/R_$(r)/$(r)-year$(y)pred_VAF.jld2"
        isfile(f) && push!(ms, mean(JLD2.load(f, "sample_post_mm_ssp5")[hrow(h), :]))
    end
    isempty(ms) ? NaN : mean(ms)
end
@printf("\nexamples:  q width@2195 = %.4g    q fold = %.1f×    SLE-2200 width@2090 = %.3f mm\n",
        param_width(2, 2195), param_fold(2), sle_width(2200, 2090))
@printf("SLE-2100 best estimate@2090 = %.2f mm\n", sle_best(2100, 2090))
@printf("SLE-2100 best estimate@2090 = %.2f mm\n", sle_best(2200, 2195))
@printf("SLE-2100 best estimate@2090 = %.2f mm\n", sle_best(2300, 2300))

## SLE best estimate — ensemble vs present-day baseline (with sampling SE)
`sle_best(h,y)` is an average of only **100** per-trajectory means, so it carries sampling noise ≈ SD/√100. This cell reports it with that SE alongside the better-resolved **5000-draw present-day (2015) baseline** projection (SE ≈ SD/√5000, ≈7× tighter). Both estimate the ensemble-mean SLE; at the final analysis year they agree within the 100-sample noise — useful for judging how precise a quoted "best estimate" (e.g. the ≈ −0.4 mm 2100 value) really is. Runs on the current `CONFIG` (needs cell 1 + this configuration's `pred_VAF.jld2` projections).

In [ ]:
# ---- SLE "best estimate": 100-trajectory ensemble  vs  5000-draw present-day baseline ----
# `sle_best` (above) averages only 100 per-trajectory means → sampling SE ≈ SD/√100. The present-day (2015)
# baseline projection uses 5000 draws → SE ≈ SD/√5000 (≈7× tighter). Both estimate the ensemble-mean SLE;
# at the final analysis year they agree within the 100-sample noise. SE reported = SD/√n (analytic).
codeflag = occursin("nocode", SUFFIX) ? "codeoff" : "codeon"                 # A/B → codeon ; C/D → codeoff
baseline2015 = JLD2.load("../inputs/params/2015_SLR_Projections_$(codeflag).jld2", "present2015_post_mm_ssp5")  # 285×5000
_calrange(py) = collect(range(2015, step=15, length=fld(py - 2015, 15) + 1))
_finalyr(py)  = _calrange(py)[end]                                            # last analysis year ≤ horizon

# 100-trajectory ensemble best estimate at analysis year y → (mean, SE=SD/√n, n)
function ensemble_best(h, y)
    ms = Float64[]
    for r in REAL
        f = "$PROJ/R_$(r)/$(r)-year$(y)pred_VAF.jld2"
        isfile(f) && push!(ms, mean(JLD2.load(f, "sample_post_mm_ssp5")[hrow(h), :]))
    end
    isempty(ms) ? (NaN, NaN, 0) : (mean(ms), std(ms)/sqrt(length(ms)), length(ms))
end
# 5000-draw present-day (2015) baseline best estimate → (mean, SE=SD/√n, n)
baseline_best(h) = (v = baseline2015[hrow(h), :]; (mean(v), std(v)/sqrt(length(v)), length(v)))

println("CONFIG = $CONFIG   ($SUFFIX)")
println("SEA-LEVEL BEST ESTIMATE (mm SLE):  100-traj ensemble (SE=SD/√100)  vs  5000-draw present-day baseline (SE=SD/√5000)")
for py in PRED
    y = _finalyr(py)
    em, es, en = ensemble_best(py, y);  bm, bs, bn = baseline_best(py)
    @printf("  %-4d @ %-4d :  ensemble %8.3f ± %.3f (n=%d)   |   baseline %8.3f ± %.3f (n=%d)   |   Δ = %+.3f\n",
            py, y, em, es, en, bm, bs, bn, em - bm)
end

## All four configurations at once
Self-contained — independent of the `CONFIG` switch above (loads every configuration's caches directly). The four-configuration %reduction table, cross-configuration ratios, and SLE folds in one shot.


In [ ]:
# ---- ALL-FOUR-CONFIG comparison (self-contained) ----
_pn4  = ["σ_max", "q", "C_μ", "C_φ", "log(γ₀)", "m̄"]
_PRED = [2100, 2200, 2300]
_SC   = [("A indep_code", "_indep_code"),   ("B  wiener_code", "_wiener_code"),
         ("C  indep_nocode", "_indep_nocode"), ("D  wiener_nocode", "_wiener_nocode")]

_final = Dict{String,Vector{Float64}}();  _red = Dict{String,Vector{Float64}}();  _slef = Dict{String,Dict{Int,Float64}}()
for (_, suf) in _SC
    tw = JLD2.load("../outputs/posteriors$(suf)/total_widths.jld2", "total_widths")
    MW = [ mean(filter(!isnan, tw[:, i, j])) for i in 1:size(tw, 2), j in 1:6 ]
    _final[suf] = MW[end, :];  _red[suf] = [100*(1 - MW[end, j]/MW[1, j]) for j in 1:6]
    slr = JLD2.load("../outputs/projections$(suf)/slr_learning_widths.jld2")
    pw, sw = slr["present_width"], slr["slr_widths"]
    _slef[suf] = Dict(py => pw[py] / mean(filter(!isnan, sw[py][:, end])) for py in _PRED)
end

println("PARAMETER %reduction by 2300")
@printf("%-9s %9s %9s %9s %9s\n", "param", "A", "B", "C", "D")
for j in 1:6
    @printf("%-9s %8.1f%% %8.1f%% %8.1f%% %8.1f%%\n", _pn4[j],
            _red["_indep_code"][j], _red["_wiener_code"][j], _red["_indep_nocode"][j], _red["_wiener_nocode"][j])
end

println("\nFinal-2300 width RATIOS   (<1 ⇒ numerator configuration learns more)")
@printf("%-9s %13s %13s %9s %9s\n", "param", "B/A codeon", "D/C codeoff", "C/A", "D/B")
for j in 1:6
    a=_final["_indep_code"][j]; b=_final["_wiener_code"][j]; c=_final["_indep_nocode"][j]; d=_final["_wiener_nocode"][j]
    @printf("%-9s %13.3f %13.3f %9.3f %9.3f\n", _pn4[j], b/a, d/c, c/a, d/b)
end

println("\nSLE fold by horizon")
@printf("%-8s %9s %9s %9s %9s\n", "horizon", "A", "B", "C", "D")
for py in _PRED
    @printf("%-8d %9.2f %9.2f %9.2f %9.2f\n", py,
            _slef["_indep_code"][py], _slef["_wiener_code"][py], _slef["_indep_nocode"][py], _slef["_wiener_nocode"][py])
end